# 📊 Data Understanding & Exploratory Analysis

## Supply Chain Late Delivery Prediction

---

### 🎯 Business Problem

**Objective**: Build a machine learning model to predict late deliveries in a supply chain network, enabling proactive customer communication and operational interventions.

**Business Value**:
- Reduce customer complaints from unexpected delays
- Enable proactive shipping upgrades for high-risk orders
- Improve customer satisfaction and retention
- Optimize logistics resource allocation

---

### 📋 Dataset Overview

**Source**: DataCo Supply Chain Dataset (180K+ e-commerce transactions)

| Category | Features | Description |
|----------|----------|-------------|
| **Transaction** | Type, Order Status | Transaction type and order processing status |
| **Delivery** | Days for shipping (real/scheduled), Delivery Status, Late_delivery_risk | Actual vs planned shipping, delivery outcomes |
| **Financial** | Sales, Benefit per order, Order Profit, Discount | Revenue and profitability metrics |
| **Customer** | ID, Name, Segment, Location (City/State/Country) | Customer demographics and segmentation |
| **Product** | Category, Name, Price, Status | Product catalog information |
| **Geographic** | Market, Region, Latitude/Longitude | Global distribution network |
| **Shipping** | Shipping Mode, Shipping Date | Logistics and fulfillment details |

**Target Variable**: `Late_delivery_risk` (1 = Late, 0 = On-time)

---

### ⚠️ Critical Data Leakage Warning

The following columns contain **post-delivery information** and must be excluded from features:

| Column | Reason |
|--------|--------|
| `Late_delivery_risk` | This IS the target variable |
| `Delivery Status` | Categorical form of the target |
| `Days for shipping (real)` | Only known AFTER delivery completes |
| `Shipping date (DateOrders)` | Actual shipping timestamp (post-hoc) |

---

In [ ]:
# ============================================================
# SETUP & DATA LOADING
# ============================================================
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data.data_manager import load_raw

# Plotly defaults for professional presentation
import plotly.io as pio
pio.templates.default = "plotly_white"

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)

print("✅ Libraries loaded successfully")

In [ ]:
# Load raw data
df = load_raw()

# Standardize column names for easier handling
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

print(f"📊 Dataset loaded: {df.shape[0]:,} orders × {df.shape[1]} features")
print(f"📅 Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

---

## 1. Dataset Structure & Schema

**🎤 Talking Point**: "Let me walk you through the dataset structure. We have 53 features across transaction, customer, product, and logistics dimensions."

In [ ]:
# ============================================================
# DATASET SCHEMA ANALYSIS
# ============================================================

# Analyze data types
dtype_counts = df.dtypes.value_counts()
schema_info = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null %': (df.isnull().sum() / len(df) * 100).round(2).values,
    'Unique Values': df.nunique().values,
    'Sample Value': [df[col].dropna().iloc[0] if df[col].notnull().any() else 'N/A' for col in df.columns]
})

# Display schema summary
print("📋 DATASET SCHEMA SUMMARY")
print("=" * 60)
print(f"\nTotal Features: {len(df.columns)}")
print(f"Total Records: {len(df):,}")
print(f"\nData Types Distribution:")
for dtype, count in dtype_counts.items():
    print(f"   {dtype}: {count} columns")

# Display full schema
schema_info

In [ ]:
# Visualize data types distribution
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=('<b>Data Types Distribution</b>', '<b>Missing Values by Column</b>')
)

# Pie chart for data types
fig.add_trace(
    go.Pie(
        labels=dtype_counts.index.astype(str),
        values=dtype_counts.values,
        hole=0.4,
        marker_colors=px.colors.qualitative.Set2
    ),
    row=1, col=1
)

# Bar chart for missing values (only columns with missing values)
missing_df = schema_info[schema_info['Null %'] > 0].sort_values('Null %', ascending=True)
if len(missing_df) > 0:
    fig.add_trace(
        go.Bar(
            x=missing_df['Null %'].values,
            y=missing_df['Column'].values,
            orientation='h',
            marker_color='#e74c3c',
            text=[f"{v:.1f}%" for v in missing_df['Null %'].values],
            textposition='outside'
        ),
        row=1, col=2
    )

fig.update_layout(
    height=400,
    title_text='<b>Dataset Structure Overview</b>',
    showlegend=False
)
fig.update_xaxes(title_text="Missing %", row=1, col=2)
fig.show()

# Dynamic interpretation
total_missing = df.isnull().sum().sum()
missing_cols = (df.isnull().sum() > 0).sum()
high_missing_cols = schema_info[schema_info['Null %'] > 50]['Column'].tolist()

print(f"\n📊 DYNAMIC INTERPRETATION:")
print(f"   • Total missing values: {total_missing:,} ({total_missing/(len(df)*len(df.columns))*100:.2f}% of all cells)")
print(f"   • Columns with missing data: {missing_cols} out of {len(df.columns)}")
if high_missing_cols:
    print(f"   • High missing (>50%): {', '.join(high_missing_cols)}")
print(f"   • Recommendation: Drop 'product_description' (100% null), impute others")

---

## 2. Target Variable Analysis: Late Delivery Risk

**🎤 Talking Point**: "Our target is binary classification - predicting whether an order will be delivered late. The dataset shows a slight class imbalance with ~55% late deliveries."

In [ ]:
# ============================================================
# TARGET VARIABLE ANALYSIS
# ============================================================

# Find target column (handle different naming conventions)
target_col = None
for col in ['late_delivery_risk', 'late_delivery', 'latedeliveryrisk']:
    if col in df.columns:
        target_col = col
        break

if target_col:
    target_dist = df[target_col].value_counts()
    target_pct = df[target_col].value_counts(normalize=True) * 100
    
    # Also analyze delivery_status if available
    delivery_status_col = [c for c in df.columns if 'delivery_status' in c.lower()]
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "pie"}, {"type": "bar"}]],
        subplot_titles=('<b>Late Delivery Risk Distribution</b>', '<b>Delivery Status Breakdown</b>')
    )
    
    # Target variable pie chart
    labels = ['Late Delivery' if v == 1 else 'On-Time' for v in target_dist.index]
    colors = ['#e74c3c', '#2ecc71']
    
    fig.add_trace(
        go.Pie(
            labels=labels,
            values=target_dist.values,
            hole=0.4,
            marker_colors=colors,
            textinfo='percent+label',
            textfont_size=14
        ),
        row=1, col=1
    )
    
    # Delivery status bar chart
    if delivery_status_col:
        status_dist = df[delivery_status_col[0]].value_counts()
        status_colors = {'Late delivery': '#e74c3c', 'Advance shipping': '#2ecc71', 
                        'Shipping on time': '#3498db', 'Shipping canceled': '#95a5a6'}
        bar_colors = [status_colors.get(s, '#7f8c8d') for s in status_dist.index]
        
        fig.add_trace(
            go.Bar(
                x=status_dist.index,
                y=status_dist.values,
                marker_color=bar_colors,
                text=[f"{v:,}" for v in status_dist.values],
                textposition='outside'
            ),
            row=1, col=2
        )
    
    fig.update_layout(
        height=450,
        title_text='<b>Target Variable: Late Delivery Analysis</b>',
        showlegend=False
    )
    fig.update_xaxes(tickangle=45, row=1, col=2)
    fig.update_yaxes(title_text="Order Count", row=1, col=2)
    fig.show()
    
    # Dynamic interpretation
    late_count = target_dist.get(1, 0)
    ontime_count = target_dist.get(0, 0)
    late_pct = target_pct.get(1, 0)
    imbalance_ratio = max(late_count, ontime_count) / min(late_count, ontime_count)
    
    print(f"\n📊 TARGET VARIABLE STATISTICS:")
    print(f"   • Late deliveries: {late_count:,} ({late_pct:.1f}%)")
    print(f"   • On-time deliveries: {ontime_count:,} ({100-late_pct:.1f}%)")
    print(f"   • Class imbalance ratio: {imbalance_ratio:.2f}:1")
    
    if imbalance_ratio < 1.5:
        balance_status = "✅ Classes are relatively balanced - standard algorithms should work well"
    elif imbalance_ratio < 3:
        balance_status = "⚠️ Moderate imbalance - consider class weights or stratified sampling"
    else:
        balance_status = "❌ Severe imbalance - use SMOTE, class weights, or PR-AUC metric"
    
    print(f"   • {balance_status}")
else:
    print("⚠️ Target variable not found in dataset")

---

## 3. Shipping & Logistics Features

**🎤 Talking Point**: "Shipping mode and scheduled delivery days are among the strongest predictors. Let's examine how different shipping options correlate with late deliveries."

### Feature Definitions:
- **Days for shipping (real)**: Actual shipping days (⚠️ LEAKY - only known after delivery)
- **Days for shipment (scheduled)**: Promised delivery timeframe (✅ SAFE for features)
- **Shipping Mode**: Standard Class, First Class, Second Class, Same Day

In [ ]:
# ============================================================
# SHIPPING & LOGISTICS ANALYSIS
# ============================================================

# Find shipping-related columns
shipping_mode_col = [c for c in df.columns if 'shipping_mode' in c.lower()]
days_scheduled_col = [c for c in df.columns if 'scheduled' in c.lower()]
days_real_col = [c for c in df.columns if 'real' in c.lower() and 'day' in c.lower()]

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "box"}]],
    subplot_titles=(
        '<b>Orders by Shipping Mode</b>',
        '<b>Late Delivery Rate by Shipping Mode</b>',
        '<b>Scheduled Shipping Days Distribution</b>',
        '<b>Actual vs Scheduled Days</b>'
    )
)

# 1. Shipping mode distribution
if shipping_mode_col:
    mode_col = shipping_mode_col[0]
    mode_dist = df[mode_col].value_counts()
    mode_colors = {'Same Day': '#27ae60', 'First Class': '#3498db', 
                   'Second Class': '#f39c12', 'Standard Class': '#e74c3c'}
    
    fig.add_trace(
        go.Bar(
            x=mode_dist.index,
            y=mode_dist.values,
            marker_color=[mode_colors.get(m, '#7f8c8d') for m in mode_dist.index],
            text=[f"{v:,}" for v in mode_dist.values],
            textposition='outside',
            name='Orders'
        ),
        row=1, col=1
    )
    
    # 2. Late delivery rate by shipping mode
    if target_col:
        late_by_mode = df.groupby(mode_col)[target_col].mean() * 100
        late_by_mode = late_by_mode.sort_values(ascending=False)
        
        fig.add_trace(
            go.Bar(
                x=late_by_mode.index,
                y=late_by_mode.values,
                marker_color=[mode_colors.get(m, '#7f8c8d') for m in late_by_mode.index],
                text=[f"{v:.1f}%" for v in late_by_mode.values],
                textposition='outside',
                name='Late Rate'
            ),
            row=1, col=2
        )
        
        # Add average line
        avg_late = df[target_col].mean() * 100
        fig.add_hline(y=avg_late, line_dash="dash", line_color="red",
                      annotation_text=f"Avg: {avg_late:.1f}%", row=1, col=2)

# 3. Scheduled days distribution
if days_scheduled_col:
    sched_col = days_scheduled_col[0]
    fig.add_trace(
        go.Histogram(
            x=df[sched_col].dropna(),
            nbinsx=20,
            marker_color='#3498db',
            name='Scheduled Days'
        ),
        row=2, col=1
    )

# 4. Actual vs Scheduled days boxplot
if days_scheduled_col and days_real_col:
    sched_col = days_scheduled_col[0]
    real_col = days_real_col[0]
    
    fig.add_trace(
        go.Box(y=df[sched_col].dropna(), name='Scheduled', marker_color='#3498db'),
        row=2, col=2
    )
    fig.add_trace(
        go.Box(y=df[real_col].dropna(), name='Actual', marker_color='#e74c3c'),
        row=2, col=2
    )

fig.update_layout(height=700, showlegend=False,
                  title_text='<b>Shipping & Logistics Analysis</b>')
fig.update_yaxes(title_text="Order Count", row=1, col=1)
fig.update_yaxes(title_text="Late Rate (%)", row=1, col=2)
fig.update_xaxes(title_text="Days", row=2, col=1)
fig.show()

# Dynamic interpretation
if shipping_mode_col and target_col:
    best_mode = late_by_mode.idxmin()
    worst_mode = late_by_mode.idxmax()
    best_rate = late_by_mode.min()
    worst_rate = late_by_mode.max()
    
    print(f"\n📊 SHIPPING ANALYSIS - KEY FINDINGS:")
    print(f"   • Best performing: {best_mode} ({best_rate:.1f}% late rate)")
    print(f"   • Worst performing: {worst_mode} ({worst_rate:.1f}% late rate)")
    print(f"   • Difference: {worst_rate - best_rate:.1f} percentage points")
    print(f"   • 🎯 Recommendation: Shipping mode is a STRONG predictor - include in features")

---

## 4. Financial Features Analysis

**🎤 Talking Point**: "Financial metrics like sales, profit margins, and discounts help us understand the business impact of late deliveries and identify high-value orders requiring priority handling."

### Feature Definitions:
- **Sales**: Total sales value of the order
- **Benefit per order**: Earnings/profit per order
- **Order Profit Per Order**: Net profit after costs
- **Order Item Discount / Discount Rate**: Discount applied to items

In [ ]:
# ============================================================
# FINANCIAL FEATURES ANALYSIS
# ============================================================

# Identify financial columns
financial_cols = [c for c in df.columns if any(term in c.lower() for term in 
                  ['sales', 'profit', 'benefit', 'discount', 'price', 'total'])]

# Select key financial metrics
key_financial = []
for term in ['sales', 'order_profit', 'benefit', 'discount_rate', 'order_item_total']:
    matches = [c for c in df.columns if term in c.lower()]
    if matches:
        key_financial.append(matches[0])

key_financial = list(set(key_financial))[:4]  # Limit to 4

if key_financial:
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[f'<b>{col.replace("_", " ").title()}</b>' for col in key_financial[:4]]
    )
    
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
    
    for i, col in enumerate(key_financial[:4]):
        row = (i // 2) + 1
        col_num = (i % 2) + 1
        
        data = df[col].dropna()
        
        # Use log scale for highly skewed distributions
        if data.skew() > 2:
            data_plot = np.log1p(data[data > 0])
            x_title = f"Log({col})"
        else:
            data_plot = data
            x_title = col
        
        fig.add_trace(
            go.Histogram(
                x=data_plot,
                nbinsx=50,
                marker_color=colors[i],
                opacity=0.8,
                name=col
            ),
            row=row, col=col_num
        )
        
        # Add median line
        median_val = data.median()
        fig.add_vline(
            x=np.log1p(median_val) if data.skew() > 2 and median_val > 0 else median_val,
            line_dash="dash", line_color="red",
            annotation_text=f"Median: {median_val:.2f}",
            row=row, col=col_num
        )
    
    fig.update_layout(height=600, showlegend=False,
                      title_text='<b>Financial Features Distribution</b>')
    fig.show()
    
    # Summary statistics
    print("\n📊 FINANCIAL FEATURES SUMMARY:")
    print("=" * 70)
    financial_stats = df[key_financial].describe().T
    financial_stats['skewness'] = df[key_financial].skew()
    print(financial_stats.round(2).to_string())

In [ ]:
# Financial features vs Late Delivery
if target_col and key_financial:
    fig = make_subplots(
        rows=1, cols=len(key_financial[:3]),
        subplot_titles=[f'<b>{c.replace("_", " ").title()} by Delivery Status</b>' for c in key_financial[:3]]
    )
    
    for i, col in enumerate(key_financial[:3]):
        for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
            subset = df[df[target_col] == label][col].dropna()
            fig.add_trace(
                go.Box(
                    y=subset,
                    name='Late' if label == 1 else 'On-Time',
                    marker_color=color,
                    showlegend=(i == 0)
                ),
                row=1, col=i+1
            )
    
    fig.update_layout(height=400, title_text='<b>Financial Metrics: Late vs On-Time Deliveries</b>')
    fig.show()
    
    # Calculate mean differences
    print("\n📊 FINANCIAL IMPACT OF LATE DELIVERIES:")
    for col in key_financial[:3]:
        late_mean = df[df[target_col] == 1][col].mean()
        ontime_mean = df[df[target_col] == 0][col].mean()
        diff_pct = ((late_mean - ontime_mean) / ontime_mean * 100) if ontime_mean != 0 else 0
        print(f"   • {col}: Late avg={late_mean:.2f}, On-time avg={ontime_mean:.2f} ({diff_pct:+.1f}%)")

---

## 5. Customer Segmentation Analysis

**🎤 Talking Point**: "Understanding customer segments helps prioritize which late deliveries have the highest business impact. Corporate customers may have different risk tolerance than consumers."

### Feature Definitions:
- **Customer Segment**: Consumer, Corporate, Home Office
- **Customer City/State/Country**: Geographic location
- **Sales per customer**: Total historical purchases

In [ ]:
# ============================================================
# CUSTOMER SEGMENTATION ANALYSIS
# ============================================================

segment_col = [c for c in df.columns if 'segment' in c.lower()]

if segment_col:
    seg_col = segment_col[0]
    segment_dist = df[seg_col].value_counts()
    segment_colors = {'Consumer': '#3498db', 'Corporate': '#2ecc71', 'Home Office': '#e74c3c'}
    
    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "pie"}, {"type": "bar"}],
               [{"type": "bar"}, {"type": "bar"}]],
        subplot_titles=(
            '<b>Customer Segment Distribution</b>',
            '<b>Late Delivery Rate by Segment</b>',
            '<b>Average Order Value by Segment</b>',
            '<b>Total Revenue by Segment</b>'
        )
    )
    
    # 1. Segment distribution pie
    fig.add_trace(
        go.Pie(
            labels=segment_dist.index,
            values=segment_dist.values,
            hole=0.4,
            marker_colors=[segment_colors.get(s, '#7f8c8d') for s in segment_dist.index],
            textinfo='percent+label'
        ),
        row=1, col=1
    )
    
    # 2. Late rate by segment
    if target_col:
        late_by_seg = df.groupby(seg_col)[target_col].mean() * 100
        fig.add_trace(
            go.Bar(
                x=late_by_seg.index,
                y=late_by_seg.values,
                marker_color=[segment_colors.get(s, '#7f8c8d') for s in late_by_seg.index],
                text=[f"{v:.1f}%" for v in late_by_seg.values],
                textposition='outside'
            ),
            row=1, col=2
        )
        fig.add_hline(y=df[target_col].mean()*100, line_dash="dash", line_color="red",
                      annotation_text="Overall Avg", row=1, col=2)
    
    # 3. Average order value by segment
    sales_col = [c for c in df.columns if c.lower() == 'sales']
    if sales_col:
        avg_sales = df.groupby(seg_col)[sales_col[0]].mean()
        fig.add_trace(
            go.Bar(
                x=avg_sales.index,
                y=avg_sales.values,
                marker_color=[segment_colors.get(s, '#7f8c8d') for s in avg_sales.index],
                text=[f"${v:.0f}" for v in avg_sales.values],
                textposition='outside'
            ),
            row=2, col=1
        )
        
        # 4. Total revenue by segment
        total_sales = df.groupby(seg_col)[sales_col[0]].sum()
        fig.add_trace(
            go.Bar(
                x=total_sales.index,
                y=total_sales.values,
                marker_color=[segment_colors.get(s, '#7f8c8d') for s in total_sales.index],
                text=[f"${v/1e6:.1f}M" for v in total_sales.values],
                textposition='outside'
            ),
            row=2, col=2
        )
    
    fig.update_layout(height=700, showlegend=False,
                      title_text='<b>Customer Segment Analysis</b>')
    fig.update_yaxes(title_text="Late Rate (%)", row=1, col=2)
    fig.update_yaxes(title_text="Avg Order Value ($)", row=2, col=1)
    fig.update_yaxes(title_text="Total Revenue ($)", row=2, col=2)
    fig.show()
    
    # Dynamic interpretation
    print("\n📊 CUSTOMER SEGMENT INSIGHTS:")
    print(f"   • Largest segment: {segment_dist.idxmax()} ({segment_dist.max():,} orders, {segment_dist.max()/len(df)*100:.1f}%)")
    if target_col:
        print(f"   • Highest late rate: {late_by_seg.idxmax()} ({late_by_seg.max():.1f}%)")
        print(f"   • Lowest late rate: {late_by_seg.idxmin()} ({late_by_seg.min():.1f}%)")
        rate_spread = late_by_seg.max() - late_by_seg.min()
        if rate_spread < 2:
            print(f"   • ⚠️ Late rate is consistent across segments (spread: {rate_spread:.1f}pp) - segment alone is weak predictor")
        else:
            print(f"   • ✅ Significant variation across segments (spread: {rate_spread:.1f}pp) - include in features")

---

## 6. Geographic & Market Analysis

**🎤 Talking Point**: "Geographic patterns reveal regional logistics challenges. Some markets may have inherently higher late delivery rates due to infrastructure or distance factors."

### Feature Definitions:
- **Market**: Africa, Europe, LATAM, Pacific Asia, USCA
- **Order Region**: Southeast Asia, South Asia, Oceania, Eastern Europe, etc.
- **Order Country/State/City**: Specific delivery location
- **Latitude/Longitude**: Coordinates for distance calculations

In [ ]:
# ============================================================
# GEOGRAPHIC & MARKET ANALYSIS
# ============================================================

market_col = [c for c in df.columns if c.lower() == 'market']
region_col = [c for c in df.columns if 'order_region' in c.lower()]
country_col = [c for c in df.columns if 'order_country' in c.lower()]

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]],
    subplot_titles=(
        '<b>Orders by Market</b>',
        '<b>Late Delivery Rate by Market</b>',
        '<b>Top 10 Countries by Volume</b>',
        '<b>Top 10 Countries by Late Rate</b>'
    )
)

# 1. Market distribution
if market_col:
    m_col = market_col[0]
    market_dist = df[m_col].value_counts()
    
    fig.add_trace(
        go.Bar(
            x=market_dist.index,
            y=market_dist.values,
            marker_color=px.colors.qualitative.Set2[:len(market_dist)],
            text=[f"{v:,}" for v in market_dist.values],
            textposition='outside'
        ),
        row=1, col=1
    )
    
    # 2. Late rate by market
    if target_col:
        late_by_market = df.groupby(m_col)[target_col].mean() * 100
        late_by_market = late_by_market.sort_values(ascending=False)
        
        fig.add_trace(
            go.Bar(
                x=late_by_market.index,
                y=late_by_market.values,
                marker_color=px.colors.sequential.Reds[::-1][:len(late_by_market)],
                text=[f"{v:.1f}%" for v in late_by_market.values],
                textposition='outside'
            ),
            row=1, col=2
        )
        fig.add_hline(y=df[target_col].mean()*100, line_dash="dash", line_color="red",
                      annotation_text="Overall Avg", row=1, col=2)

# 3. Top countries by volume
if country_col:
    c_col = country_col[0]
    country_dist = df[c_col].value_counts().head(10)
    
    fig.add_trace(
        go.Bar(
            y=country_dist.index,
            x=country_dist.values,
            orientation='h',
            marker_color='#3498db',
            text=[f"{v:,}" for v in country_dist.values],
            textposition='outside'
        ),
        row=2, col=1
    )
    
    # 4. Top countries by late rate (minimum 100 orders)
    if target_col:
        country_counts = df[c_col].value_counts()
        valid_countries = country_counts[country_counts >= 100].index
        late_by_country = df[df[c_col].isin(valid_countries)].groupby(c_col)[target_col].mean() * 100
        late_by_country = late_by_country.sort_values(ascending=False).head(10)
        
        fig.add_trace(
            go.Bar(
                y=late_by_country.index,
                x=late_by_country.values,
                orientation='h',
                marker_color='#e74c3c',
                text=[f"{v:.1f}%" for v in late_by_country.values],
                textposition='outside'
            ),
            row=2, col=2
        )

fig.update_layout(height=700, showlegend=False,
                  title_text='<b>Geographic & Market Analysis</b>')
fig.update_yaxes(categoryorder='total ascending', row=2, col=1)
fig.update_yaxes(categoryorder='total ascending', row=2, col=2)
fig.show()

# Dynamic interpretation
if market_col and target_col:
    print("\n📊 GEOGRAPHIC INSIGHTS:")
    print(f"   • Largest market: {market_dist.idxmax()} ({market_dist.max():,} orders)")
    print(f"   • Highest late rate market: {late_by_market.idxmax()} ({late_by_market.max():.1f}%)")
    print(f"   • Lowest late rate market: {late_by_market.idxmin()} ({late_by_market.min():.1f}%)")
    if country_col:
        print(f"   • Countries served: {df[c_col].nunique()}")

---

## 7. Product Category Analysis

**🎤 Talking Point**: "Product characteristics can influence delivery timing. Bulky items or products requiring special handling may have different late delivery patterns."

### Feature Definitions:
- **Category Name**: Product category description
- **Department Name**: Department within the store
- **Product Price**: Unit price of the product
- **Product Status**: Stock availability (1=unavailable, 0=available)

In [ ]:
# ============================================================
# PRODUCT CATEGORY ANALYSIS
# ============================================================

category_col = [c for c in df.columns if 'category_name' in c.lower()]
department_col = [c for c in df.columns if 'department_name' in c.lower()]

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]],
    subplot_titles=(
        '<b>Top 10 Product Categories</b>',
        '<b>Late Rate by Top Categories</b>',
        '<b>Orders by Department</b>',
        '<b>Late Rate by Department</b>'
    )
)

# 1 & 2. Category analysis
if category_col:
    cat_col = category_col[0]
    cat_dist = df[cat_col].value_counts().head(10)
    
    fig.add_trace(
        go.Bar(
            y=cat_dist.index,
            x=cat_dist.values,
            orientation='h',
            marker_color='#3498db',
            text=[f"{v:,}" for v in cat_dist.values],
            textposition='outside'
        ),
        row=1, col=1
    )
    
    if target_col:
        # Filter to top categories
        top_cats = cat_dist.index.tolist()
        late_by_cat = df[df[cat_col].isin(top_cats)].groupby(cat_col)[target_col].mean() * 100
        late_by_cat = late_by_cat.reindex(top_cats)
        
        fig.add_trace(
            go.Bar(
                y=late_by_cat.index,
                x=late_by_cat.values,
                orientation='h',
                marker_color='#e74c3c',
                text=[f"{v:.1f}%" for v in late_by_cat.values],
                textposition='outside'
            ),
            row=1, col=2
        )

# 3 & 4. Department analysis
if department_col:
    dept_col = department_col[0]
    dept_dist = df[dept_col].value_counts()
    
    fig.add_trace(
        go.Bar(
            x=dept_dist.index,
            y=dept_dist.values,
            marker_color='#2ecc71',
            text=[f"{v:,}" for v in dept_dist.values],
            textposition='outside'
        ),
        row=2, col=1
    )
    
    if target_col:
        late_by_dept = df.groupby(dept_col)[target_col].mean() * 100
        late_by_dept = late_by_dept.sort_values(ascending=False)
        
        fig.add_trace(
            go.Bar(
                x=late_by_dept.index,
                y=late_by_dept.values,
                marker_color='#e74c3c',
                text=[f"{v:.1f}%" for v in late_by_dept.values],
                textposition='outside'
            ),
            row=2, col=2
        )
        fig.add_hline(y=df[target_col].mean()*100, line_dash="dash", line_color="blue",
                      annotation_text="Overall Avg", row=2, col=2)

fig.update_layout(height=700, showlegend=False,
                  title_text='<b>Product Category Analysis</b>')
fig.update_yaxes(categoryorder='total ascending', row=1, col=1)
fig.update_yaxes(categoryorder='total ascending', row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(tickangle=45, row=2, col=2)
fig.show()

# Dynamic interpretation
if category_col:
    print(f"\n📊 PRODUCT ANALYSIS INSIGHTS:")
    print(f"   • Total categories: {df[cat_col].nunique()}")
    print(f"   • Most common category: {cat_dist.idxmax()} ({cat_dist.max():,} orders)")
    if target_col:
        cat_rate_spread = late_by_cat.max() - late_by_cat.min()
        print(f"   • Late rate spread across top categories: {cat_rate_spread:.1f}pp")

---

## 8. Temporal Patterns Analysis

**🎤 Talking Point**: "Time-based patterns help identify seasonal effects and operational bottlenecks. Orders placed on certain days or during peak seasons may have higher late delivery risk."

### Feature Definitions:
- **order date (DateOrders)**: When the order was placed
- **Shipping date (DateOrders)**: When the order was shipped (⚠️ LEAKY)

In [ ]:
# ============================================================
# TEMPORAL PATTERNS ANALYSIS
# ============================================================

# Find date columns
date_cols = [c for c in df.columns if 'order_date' in c.lower() or 'dateorders' in c.lower()]
order_date_col = [c for c in date_cols if 'shipping' not in c.lower()]

if order_date_col:
    date_col = order_date_col[0]
    
    # Convert to datetime if not already
    if not pd.api.types.is_datetime64_any_dtype(df[date_col]):
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    
    df_dates = df[df[date_col].notna()].copy()
    df_dates['year'] = df_dates[date_col].dt.year
    df_dates['month'] = df_dates[date_col].dt.month
    df_dates['day_of_week'] = df_dates[date_col].dt.day_name()
    df_dates['quarter'] = df_dates[date_col].dt.quarter
    df_dates['year_month'] = df_dates[date_col].dt.to_period('M').astype(str)
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            '<b>Orders Over Time (Monthly)</b>',
            '<b>Late Rate by Month</b>',
            '<b>Orders by Day of Week</b>',
            '<b>Late Rate by Day of Week</b>'
        )
    )
    
    # 1. Monthly orders trend
    monthly_orders = df_dates.groupby('year_month').size()
    fig.add_trace(
        go.Scatter(
            x=monthly_orders.index,
            y=monthly_orders.values,
            mode='lines+markers',
            line=dict(color='#3498db', width=2),
            name='Orders'
        ),
        row=1, col=1
    )
    
    # 2. Monthly late rate
    if target_col:
        monthly_late = df_dates.groupby('year_month')[target_col].mean() * 100
        fig.add_trace(
            go.Scatter(
                x=monthly_late.index,
                y=monthly_late.values,
                mode='lines+markers',
                line=dict(color='#e74c3c', width=2),
                name='Late Rate'
            ),
            row=1, col=2
        )
    
    # 3. Day of week distribution
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    dow_dist = df_dates['day_of_week'].value_counts().reindex(day_order)
    
    fig.add_trace(
        go.Bar(
            x=dow_dist.index,
            y=dow_dist.values,
            marker_color='#3498db',
            text=[f"{v:,}" for v in dow_dist.values],
            textposition='outside'
        ),
        row=2, col=1
    )
    
    # 4. Late rate by day of week
    if target_col:
        dow_late = df_dates.groupby('day_of_week')[target_col].mean() * 100
        dow_late = dow_late.reindex(day_order)
        
        fig.add_trace(
            go.Bar(
                x=dow_late.index,
                y=dow_late.values,
                marker_color='#e74c3c',
                text=[f"{v:.1f}%" for v in dow_late.values],
                textposition='outside'
            ),
            row=2, col=2
        )
        fig.add_hline(y=df[target_col].mean()*100, line_dash="dash", line_color="blue",
                      annotation_text="Overall Avg", row=2, col=2)
    
    fig.update_layout(height=700, showlegend=False,
                      title_text='<b>Temporal Patterns Analysis</b>')
    fig.update_xaxes(tickangle=45, row=1, col=1)
    fig.update_xaxes(tickangle=45, row=1, col=2)
    fig.show()
    
    # Dynamic interpretation
    print("\n📊 TEMPORAL PATTERNS INSIGHTS:")
    date_range = f"{df_dates[date_col].min().strftime('%Y-%m-%d')} to {df_dates[date_col].max().strftime('%Y-%m-%d')}"
    print(f"   • Date range: {date_range}")
    print(f"   • Busiest day: {dow_dist.idxmax()} ({dow_dist.max():,} orders)")
    if target_col:
        print(f"   • Highest late rate day: {dow_late.idxmax()} ({dow_late.max():.1f}%)")
        print(f"   • Lowest late rate day: {dow_late.idxmin()} ({dow_late.min():.1f}%)")
        dow_spread = dow_late.max() - dow_late.min()
        if dow_spread < 2:
            print(f"   • ⚠️ Day of week has minimal impact on late rate (spread: {dow_spread:.1f}pp)")
        else:
            print(f"   • ✅ Day of week shows variation (spread: {dow_spread:.1f}pp) - consider as feature")
else:
    print("⚠️ Order date column not found")

---

## 9. Order Status & Transaction Type Analysis

**🎤 Talking Point**: "Order status and transaction type provide context about the order lifecycle. Some transaction types may be more prone to delays."

### Feature Definitions:
- **Type**: Type of transaction (e.g., DEBIT, TRANSFER, PAYMENT, CASH)
- **Order Status**: COMPLETE, PENDING, CLOSED, PENDING_PAYMENT, CANCELED, PROCESSING, SUSPECTED_FRAUD

In [ ]:
# ============================================================
# ORDER STATUS & TRANSACTION TYPE ANALYSIS
# ============================================================

type_col = [c for c in df.columns if c.lower() == 'type']
status_col = [c for c in df.columns if 'order_status' in c.lower()]

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=(
        '<b>Transaction Type Distribution</b>',
        '<b>Late Rate by Transaction Type</b>',
        '<b>Order Status Distribution</b>',
        '<b>Late Rate by Order Status</b>'
    )
)

# 1 & 2. Transaction type
if type_col:
    t_col = type_col[0]
    type_dist = df[t_col].value_counts()
    
    fig.add_trace(
        go.Pie(
            labels=type_dist.index,
            values=type_dist.values,
            hole=0.4,
            marker_colors=px.colors.qualitative.Set2
        ),
        row=1, col=1
    )
    
    if target_col:
        late_by_type = df.groupby(t_col)[target_col].mean() * 100
        late_by_type = late_by_type.sort_values(ascending=False)
        
        fig.add_trace(
            go.Bar(
                x=late_by_type.index,
                y=late_by_type.values,
                marker_color='#e74c3c',
                text=[f"{v:.1f}%" for v in late_by_type.values],
                textposition='outside'
            ),
            row=1, col=2
        )

# 3 & 4. Order status
if status_col:
    s_col = status_col[0]
    status_dist = df[s_col].value_counts()
    
    fig.add_trace(
        go.Pie(
            labels=status_dist.index,
            values=status_dist.values,
            hole=0.4,
            marker_colors=px.colors.qualitative.Pastel
        ),
        row=2, col=1
    )
    
    if target_col:
        late_by_status = df.groupby(s_col)[target_col].mean() * 100
        late_by_status = late_by_status.sort_values(ascending=False)
        
        fig.add_trace(
            go.Bar(
                x=late_by_status.index,
                y=late_by_status.values,
                marker_color='#9b59b6',
                text=[f"{v:.1f}%" for v in late_by_status.values],
                textposition='outside'
            ),
            row=2, col=2
        )

fig.update_layout(height=700, showlegend=False,
                  title_text='<b>Order Status & Transaction Type Analysis</b>')
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=2)
fig.show()

# Dynamic interpretation
print("\n📊 ORDER STATUS & TRANSACTION TYPE INSIGHTS:")
if type_col:
    print(f"   • Transaction types: {df[t_col].nunique()}")
    print(f"   • Most common type: {type_dist.idxmax()} ({type_dist.max()/len(df)*100:.1f}%)")
if status_col:
    print(f"   • Order statuses: {df[s_col].nunique()}")
    print(f"   • Most common status: {status_dist.idxmax()} ({status_dist.max()/len(df)*100:.1f}%)")

---

## 10. Feature Correlation Analysis

**🎤 Talking Point**: "Correlation analysis helps identify which features are most predictive of late delivery and detect multicollinearity that could affect model performance."

In [ ]:
# ============================================================
# FEATURE CORRELATION ANALYSIS
# ============================================================

# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove ID columns and limit to key features
exclude_patterns = ['_id', 'id_', 'latitude', 'longitude', 'zipcode']
numeric_cols = [c for c in numeric_cols if not any(p in c.lower() for p in exclude_patterns)]

# Calculate correlation matrix
if len(numeric_cols) > 0:
    corr_matrix = df[numeric_cols].corr()
    
    # Correlation heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale='RdBu',
        zmid=0,
        text=np.round(corr_matrix.values, 2),
        texttemplate='%{text}',
        textfont={"size": 8},
        colorbar=dict(title="Correlation")
    ))
    
    fig.update_layout(
        title='<b>Feature Correlation Matrix</b>',
        width=900,
        height=800,
        xaxis={'tickangle': 45},
        yaxis={'autorange': 'reversed'}
    )
    fig.show()

In [ ]:
# Target correlation analysis
if target_col and target_col in numeric_cols:
    target_corr = corr_matrix[target_col].drop(target_col).sort_values(key=abs, ascending=False)
    
    # Create bar chart of correlations with target
    fig = go.Figure()
    
    colors = ['#e74c3c' if v > 0 else '#3498db' for v in target_corr.head(15).values]
    
    fig.add_trace(go.Bar(
        y=target_corr.head(15).index,
        x=target_corr.head(15).values,
        orientation='h',
        marker_color=colors,
        text=[f"{v:.3f}" for v in target_corr.head(15).values],
        textposition='outside'
    ))
    
    fig.update_layout(
        title='<b>Top 15 Features Correlated with Late Delivery</b>',
        xaxis_title='Correlation Coefficient',
        yaxis={'categoryorder': 'total ascending'},
        height=500,
        showlegend=False
    )
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.show()
    
    # Dynamic interpretation
    print("\n📊 TARGET CORRELATION INSIGHTS:")
    print(f"   • Strongest positive correlation: {target_corr.idxmax()} ({target_corr.max():.3f})")
    print(f"   • Strongest negative correlation: {target_corr.idxmin()} ({target_corr.min():.3f})")
    
    # Identify features with strong correlation (>0.1 or <-0.1)
    strong_features = target_corr[abs(target_corr) > 0.1].index.tolist()
    print(f"   • Features with |corr| > 0.1: {len(strong_features)}")
    for feat in strong_features[:5]:
        direction = "↑ late" if target_corr[feat] > 0 else "↓ late"
        print(f"      - {feat}: {target_corr[feat]:.3f} ({direction})")

---

## 11. EDA Summary & Key Findings

**🎤 Talking Point**: "Let me summarize our key findings from the exploratory analysis and outline our modeling strategy."

In [ ]:
# ============================================================
# COMPREHENSIVE EDA SUMMARY
# ============================================================

print("=" * 80)
print("📊 EXPLORATORY DATA ANALYSIS - EXECUTIVE SUMMARY")
print("=" * 80)

# Calculate key statistics dynamically
late_rate = df[target_col].mean() * 100 if target_col else 55
total_orders = len(df)
total_features = len(df.columns)
missing_pct = (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100

summary_text = f"""
📋 DATASET OVERVIEW
{'='*60}
   • Total orders: {total_orders:,}
   • Total features: {total_features}
   • Date range: {df[date_col].min().strftime('%Y-%m-%d') if order_date_col else 'N/A'} to {df[date_col].max().strftime('%Y-%m-%d') if order_date_col else 'N/A'}
   • Missing data: {missing_pct:.2f}% of all cells

🎯 TARGET VARIABLE
{'='*60}
   • Late delivery rate: {late_rate:.1f}%
   • Class balance: {'Moderate imbalance' if late_rate > 40 and late_rate < 60 else 'Imbalanced'}
   • Recommendation: {'Standard algorithms OK' if late_rate > 40 and late_rate < 60 else 'Use class weights'}

🔑 TOP PREDICTORS IDENTIFIED
{'='*60}
   1. Shipping Mode - Strong predictor (Same Day best, Standard worst)
   2. Scheduled Shipping Days - Negative correlation with late delivery
   3. Market/Region - Geographic variation in late rates
   4. Order Value/Sales - Financial characteristics

⚠️  DATA LEAKAGE PREVENTION
{'='*60}
   EXCLUDE from features (known only after delivery):
   • late_delivery_risk (target)
   • delivery_status (target in different form)
   • days_for_shipping_(real) (actual delivery time)
   • shipping_date (actual shipping timestamp)

📈 RECOMMENDED MODELING APPROACH
{'='*60}
   • Problem Type: Binary Classification
   • Primary Metric: F1 Score (balanced precision/recall)
   • Secondary Metrics: ROC-AUC, Precision, Recall
   • Algorithms: Gradient Boosting (XGBoost, CatBoost, LightGBM)
   • Validation: Stratified K-Fold Cross-Validation

{'='*80}
✅ EDA COMPLETE - Ready for Data Preprocessing
{'='*80}
➡️ Next: Run 02_data_preprocessing.ipynb
"""

print(summary_text)

In [ ]:
# Create summary dashboard
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=('Total Orders', 'Features', 'Late Rate',
                    'Markets', 'Customer Segments', 'Shipping Modes')
)

# Row 1 indicators
fig.add_trace(go.Indicator(
    mode="number",
    value=total_orders,
    number={'font': {'size': 40, 'color': '#3498db'}, 'valueformat': ','},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_features,
    number={'font': {'size': 40, 'color': '#2ecc71'}}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=late_rate,
    number={'font': {'size': 40, 'color': '#e74c3c'}, 'suffix': '%'}
), row=1, col=3)

# Row 2 indicators
fig.add_trace(go.Indicator(
    mode="number",
    value=df[market_col[0]].nunique() if market_col else 0,
    number={'font': {'size': 40, 'color': '#9b59b6'}}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=df[segment_col[0]].nunique() if segment_col else 0,
    number={'font': {'size': 40, 'color': '#f39c12'}}
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=df[shipping_mode_col[0]].nunique() if shipping_mode_col else 0,
    number={'font': {'size': 40, 'color': '#1abc9c'}}
), row=2, col=3)

fig.update_layout(
    title='<b>📊 Dataset Overview Dashboard</b>',
    height=400,
    paper_bgcolor='#f8f9fa'
)
fig.show()